# GGA AutoPILOT: Gemma 4 31B with macros and live controllers
Inputs: the competition data and gemma-4-31b-it-qat-w4a16-ct. Accelerator: RTX Pro 6000. Internet: off. Built by `make_notebook.py`, do not edit here.

In [ ]:
# Setup: installs vLLM and Chromium offline and starts Gemma 4 31B (4-bit) in a vLLM server on localhost:8000.
import io, json, os, subprocess, sys, threading, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

began = time.time()
root = Path('/kaggle/input')
harness = next(root.rglob('gga/protocol.py')).parents[1]
weights = next(root.rglob('gemma-4-31b-it-qat-w4a16-ct/*/model.safetensors')).parent
kit = next(root.rglob('kit/wheels')).parent
wheels = next(root.rglob('vllm-0.34.0-*.whl')).parent   # output of the notebook gga-vllm-wheels


def log(text):
    print(f'{time.time() - began:5.0f} s  {text}', flush=True)


# Chromium installs meanwhile. The input folder drops executable bits, so it is copied out first.
chromium = subprocess.Popen(f'cp -r {kit}/browsers /tmp && chmod -R +x /tmp/browsers'
                            f' && dpkg -i {kit}/debs/*.deb > /dev/null 2>&1', shell=True)
# The input folder is slow per stream but scales: read the weights in 16 streams, so vLLM loads from memory.
fd = os.open(weights / 'model.safetensors', os.O_RDONLY)
reading = ThreadPoolExecutor(16).map(lambda offset: os.pread(fd, 64 * 2**20, offset),
                                     range(0, os.fstat(fd).st_size, 64 * 2**20))
subprocess.run(f'pip install -q --no-index --find-links {wheels} --find-links {kit}/wheels vllm playwright',
               shell=True, check=True)
# vLLM brings torch for CUDA 13; the image's torchaudio (CUDA 12.8) then fails on import. Gemma needs no audio.
subprocess.run('pip uninstall -q -y torchaudio', shell=True, check=True)
log('installed vLLM and Playwright')
subprocess.Popen(['Xvfb', ':99', '-screen', '0', '1920x1080x24'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
sys.path.insert(0, str(harness))
sys.path.insert(0, '/kaggle/working')   # for autopilot.py, written by the next cell
os.environ.update(GGA_ROOT=str(harness), GGA_SEED='0', DISPLAY=':99', PLAYWRIGHT_BROWSERS_PATH='/tmp/browsers')
# FlashInfer's sampler compiles at startup and fails to link (-lcuda missing); greedy needs no sampler anyway
os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
list(reading)   # raises if a read failed
os.close(fd)
log('read the weights')

server = subprocess.Popen(
    ['vllm', 'serve', str(weights), '--port', '8000', '--served-model-name', 'gemma', '--max-model-len', '16384',
     '--gpu-memory-utilization', '0.9', '--max-num-seqs', '8', '--limit-mm-per-prompt', '{"image": 1}'],
    stdout=open('/kaggle/working/vllm.log', 'w'), stderr=subprocess.STDOUT)
while True:   # the competition allows 15 minutes for everything before the inference server listens
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        break
    except OSError:
        if server.poll() is not None:
            sys.exit('vLLM exited:\n' + Path('/kaggle/working/vllm.log').read_text()[-3000:])
        time.sleep(2)
log('vLLM is serving Gemma')
if chromium.wait():
    raise RuntimeError('Installing Chromium failed')
log('ready')


In [ ]:
%%writefile /kaggle/working/autopilot.py
"""AutoPILOT-style agent for the Generalist Gaming Agent competition.

step() never waits for the model. The model (`think`) runs in a background thread and is only
woken by events. Its decision is either one game action or a controller: Python code the model
writes, which then plays every decision point (200 ms) from the screen until it hands back.
"""
import io
import json
import time
from collections import deque
from concurrent.futures import ThreadPoolExecutor

import numpy as np
from PIL import Image
from gga.protocol import Action

CHECK_S = 10               # while a controller plays, the model looks at it this often
CONTROLLER_MAX_S = 0.1     # a controller call slower than this is switched off
SCENE_DIFF = 0.5           # share of pixels that must differ from ~1 s ago for a scene change
SCENE_COOLDOWN_S = 2
XY = {'x', 'y', 'x2', 'y2'}
GRID = 1000                # the model's coordinates: 0-999 over the screen, in actions and controllers alike


def to_pixels(params, width, height):
    """0-999 grid coordinates -> the game's pixels, for an action's params and its sequence steps."""
    for target in [params, *params.get('steps', [])]:
        for key, size in (('x', width), ('y', height), ('x2', width), ('y2', height)):
            if target.get(key) is not None:
                target[key] = min(max(int(target[key]), 0), GRID - 1) * (size - 1) // (GRID - 1)
    return params


class Agent:
    def __init__(self):
        self.pool = ThreadPoolExecutor(1)
        self.last_t = float('inf')
        self._reset()

    def think(self, obs, event, memory):
        """Override with the model. Returns {"tool": ..., "params": {...}}, {"controller": "<code>"},
        or {"keep": True} to let the running controller play on.

        Controller code defines control(frame, mem) -> action dict, or None to hand control back.
        frame is the screen scaled to GRID x GRID, RGB numpy, so frame[y, x] is grid point (x, y);
        mem is a dict it keeps. Its actions use grid coordinates, which the agent turns into pixels.
        To say why it hands back, it sets mem['handoff'] = '<reason>' before returning None.
        """
        return {'tool': 'skip', 'params': {}}

    def _reset(self):
        self.future = None   # shortcut: a think still running from the last game delays this game's first one
        self.memory = {}     # per game, for the model's notes
        self.controller = None
        self.controller_code = None   # its source, for the model's check-ins
        self.wake = 'new game'
        self.scenes = deque()
        self.scene_t = -SCENE_COOLDOWN_S
        self.events = []     # (game time, event) log, for debugging

    def step(self, obs, tools):
        t = obs.elapsed_s
        if t < self.last_t:   # the clock started again: a new game
            self._reset()
        self.last_t = t
        image = Image.open(io.BytesIO(obs.image)).convert('RGB')
        frame = np.asarray(image)
        self._check_scene(t, frame)
        if self.controller and obs.last_action.get('ok') is False:   # the game refused its last action
            self.controller = None
            self._wake(t, f"the game refused the controller's action: {obs.last_action.get('error')}")

        action = None
        if self.future and self.future.done():
            future, self.future = self.future, None
            try:
                action = self._apply(future.result(), t, image)
            except Exception as e:   # bad model reply: tell the model on its next call
                self._wake(t, f'your last reply failed: {e!r}')
        if action is None and self.controller:
            action = self._run_controller(t, image)
        if self.controller is None and self.future is None and not obs.program_left_s and not self.wake:
            self._wake(t, 'ready')   # nothing is playing: ask for the next move
        if self.wake and self.future is None:
            self.future = self.pool.submit(self.think, obs, self.wake, self.memory)
            self.wake = None
        return action or Action('skip')   # skip lets running inputs play on

    def _wake(self, t, event):
        self.events.append((t, event))
        self.wake = self.wake or event

    def _apply(self, decision, t, image):
        if decision.get('keep'):
            return None   # the running controller plays on
        if 'controller' in decision:
            namespace = {'np': np}
            exec(decision['controller'], namespace)   # trusted: only our own model writes this code
            self.controller, self.controller_mem = namespace['control'], {}
            self.controller_code, self.controller_start, self.controller_t = decision['controller'], t, t
            return self._run_controller(t, image)
        self.controller = None
        return Action(decision['tool'], decision.get('params', {}))

    def _run_controller(self, t, image):
        frame = np.asarray(image.resize((GRID, GRID)))
        start = time.time()
        try:
            # shortcut: an endless loop in control() hangs step(); move it to a subprocess if models write those
            choice = self.controller(frame, self.controller_mem)
            if choice is not None:   # numpy -> plain numbers, coordinates -> ints, no empty keys: as the game requires
                choice = json.loads(json.dumps(choice, default=lambda o: o.item()), object_hook=lambda d: {
                    k: round(v) if k in XY and isinstance(v, float) else [x for x in v if x != ''] if k == 'keys'
                    and isinstance(v, list) else v for k, v in d.items()})
                choice['params'] = to_pixels(choice.get('params') or {}, *image.size)
        except Exception as e:
            self.controller = None
            self._wake(t, f'controller crashed: {e!r}')
            return None
        if time.time() - start > CONTROLLER_MAX_S:
            self.controller = None
            self._wake(t, f'controller too slow ({time.time() - start:.2f} s per call), stopped')
        elif choice is None:
            self.controller = None
            self._wake(t, self.controller_mem.get('handoff', 'controller handed control back'))
            return None
        elif t - self.controller_t >= CHECK_S:
            self.controller_t = t   # it keeps playing while the model looks
            self._wake(t, f'check-in: your controller has played for {t - self.controller_start:.0f} s and plays on '
                          'while you look. Is it making progress on what counts (your game note)? '
                          'KEEP it, or reply with a better controller or an action')
        return Action(choice['tool'], choice.get('params', {}))

    def _check_scene(self, t, frame):
        small = frame[::8, ::8].mean(axis=2)
        self.scenes.append((t, small))
        while self.scenes[0][0] < t - 1:
            old = self.scenes.popleft()[1]
            if t - self.scene_t < SCENE_COOLDOWN_S or len(self.scenes) < 2:
                continue
            changed = (np.abs(small - old) > 24).mean()
            holds = (np.abs(small - self.scenes[-2][1]) > 24).mean() < 0.05
            if changed > SCENE_DIFF and holds:
                self.scene_t = t
                self.controller = None
                self._wake(t, 'the scene changed')


In [ ]:
# Notebook cell: Gemma, served by vLLM on localhost (setup cell), as the model behind autopilot.Agent.
import base64
import io
import json
import re
import time
import urllib.request

from PIL import Image

from autopilot import Agent as AutoPilot, to_pixels

SERVER = 'http://localhost:8000/v1/chat/completions'

TOOLS = '''Tools. Coordinates are on a 0-999 grid over the screen (x: left to right, y: top to bottom).
Durations are in ms, at most 10000. Keys: a printable character or a name such as ArrowLeft, Space, Enter,
Shift, Escape. Every reply except skip replaces what is still playing.
{"tool":"click","params":{"x":500,"y":500}}                     click; optional "button": "left", "middle" or "right"
{"tool":"move","params":{"x":500,"y":500}}                      move the pointer without clicking (aiming, hovering)
{"tool":"drag","params":{"x":100,"y":500,"x2":800,"y2":500,"hold_ms":100}}   press, move to x2,y2, release
{"tool":"scroll","params":{"x":500,"y":500,"dy":100}}           mouse wheel; dy > 0 down, 100 = one notch; optional dx
{"tool":"press_key","params":{"key":"Space","hold_ms":100}}     tap a key; hold it longer to keep moving
{"tool":"hold_keys","params":{"keys":["ArrowRight","Space"],"duration_ms":500}}   hold keys together, then release
{"tool":"key_down","params":{"key":"ArrowRight"}}               hold a key until key_up
{"tool":"key_up","params":{"key":"ArrowRight"}}                 release a key held by key_down
{"tool":"wait","params":{"duration_ms":500}}                    stop what is playing and idle
{"tool":"skip","params":{}}                                     change nothing: what is playing plays on
{"tool":"sequence","params":{"steps":[{"keys":["ArrowRight"],"buttons":[],"x":500,"y":500,"duration_ms":400},{"keys":["ArrowUp","x"],"buttons":["left"],"x":500,"y":500,"duration_ms":200}]}}
    a timed program of up to 200 steps that plays while you think. Each step holds exactly its keys and
    buttons (all else is released) for duration_ms; x, y are optional. The last step stays held.'''

RULES = f'''You play a real-time browser game from screenshots. The game keeps running while you think, and each
of your replies takes seconds, so prefer moves that keep playing on their own.

{TOOLS}

Start every reply with one line: Progress: <what counts in this game (your game note), as it stands on this
screen, e.g. "score 120", "alive, 3 lives", "level 2", "distance 450", "timer 1:20", or "unknown">
Then reply in exactly one of these three forms.

1. One action, as JSON on one line, from the tools above.

2. A controller: Python code that plays every 200 ms from the screen without you. Use it when the game
   needs fast reactions (dodging, aiming, steering, timing jumps).
```python
def control(frame, mem):
    # frame: numpy uint8 array (1000, 1000, 3), RGB: the screen scaled to the 0-999 grid, so frame[y, x]
    #        is grid point (x, y). np.argwhere returns (y, x). np is numpy.
    # mem: a dict kept between calls.
    # Return one action dict from the tools above, in grid coordinates as they come from frame,
    # or set mem["handoff"] = "<why>" and return None to give control back to you.
    ...
```
   Rules for controllers:
   - Use the coordinates from frame as they are: frame already is the grid, never rescale them.
   - Act on what you detect: aim at the targets you found, move away from the threats you found.
     Code that finds things and then ignores them is useless.
   - Serve what counts in this game (your game note): survive, score, go fast or get far.
   - No comments, no docstrings, no blank lines: every token you write costs game time.
   - Fast, well under 0.1 s per call: numpy color masks, never Python loops over pixels.
   - Give control back when something happens it cannot handle.

3. KEEP
   Only while your controller is playing and the progress shows it works: it plays on unchanged.
   If it is not making progress on what counts, reply with a better controller instead.

Prefer a sequence when it is enough. Test an uncertain control briefly with one action before you rely on it.
If a menu or game-over screen shows, start the game the way the brief or the screen says.
If something you tried did not help, try something different.'''

NOTE = '''A new browser game starts. Look at the screen and the brief, then write a short game note, exactly these
five lines:
Type: <the kind of game, e.g. shooter, platformer, puzzle, racing, idle, card>
Goal: <what you must do to win or do well>
Controls: <the keys and mouse actions that matter, and what each does>
Progress display: <where on screen the score, lives, level, money or timer show, on the 0-999 grid, or none>
What counts: <what to maximize: survival, points, time, distance, levels, money, or what else>'''


def parse(reply, width, height):
    """Gemma's reply -> {'controller': code}, an action dict in pixels or {'keep': True}. Raises ValueError if none."""
    code = re.search(r'```python\n(.*?)(```|$)', reply, re.S)
    if code:
        return {'controller': code.group(1)}
    start = reply.find('{')
    if start < 0:
        if re.search(r'\bKEEP\b', reply):
            return {'keep': True}
        raise ValueError('reply had no JSON action, python block or KEEP')
    choice = json.JSONDecoder().raw_decode(reply[start:])[0]
    return {'tool': choice['tool'], 'params': to_pixels(choice.get('params') or {}, width, height)}


def chat(messages, max_tokens):
    """One reply from the vLLM server: (text, tokens generated)."""
    body = json.dumps({'model': 'gemma', 'messages': messages, 'max_tokens': max_tokens, 'temperature': 0}).encode()
    request = urllib.request.Request(SERVER, body, {'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=85) as response:
        out = json.load(response)
    return out['choices'][0]['message']['content'], out['usage']['completion_tokens']


class Agent(AutoPilot):
    replies = []   # every reply of every game, for reading after a run

    def ask(self, obs, event, system, text, max_tokens):
        frame = Image.open(io.BytesIO(obs.image)).convert('RGB')
        frame.thumbnail((768, 768))
        jpeg = io.BytesIO()
        frame.save(jpeg, 'JPEG', quality=90)
        image = 'data:image/jpeg;base64,' + base64.b64encode(jpeg.getvalue()).decode()
        messages = [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': [{'type': 'image_url', 'image_url': {'url': image}}, {'type': 'text', 'text': text}]},
        ]
        began = time.time()
        reply, tokens = chat(messages, max_tokens)
        self.replies.append((obs.elapsed_s, event, reply, time.time() - began, tokens))
        return reply

    def think(self, obs, event, memory):
        info = obs.info
        brief = f"Brief: {info['description']} Controls: {info['controls']}\n"
        if 'note' not in memory:   # once per game, before its first move
            memory['note'] = self.ask(obs, 'game note', NOTE, brief, 200).strip()
        log = memory.setdefault('log', [])
        progress = memory.setdefault('progress', [])
        text = (
            brief + f"Your game note:\n{memory['note']}\n"
            f"Time left: {info['remaining_s']:.0f} s.\n"
            f"What just happened: {event}.\n"
            f"Result of the last action: {obs.last_action}\n"
            "Progress you read so far (game time: progress): "
            + (', '.join(f'{t:.0f} s: {p}' for t, p in progress[-10:]) or 'none') + '\n'
            + (f"Your controller is playing now, for {obs.elapsed_s - self.controller_start:.0f} s. Its code:\n"
               f"```python\n{self.controller_code}```\n" if self.controller else '')
            + "Your recent replies, oldest first:\n" + ('\n'.join(log[-6:]) or 'none') + '\nYour reply:'
        )
        reply = self.ask(obs, event, RULES, text, 768)
        log.append(f'[{event}] ' + ' '.join(reply.split())[:300])
        found = re.search(r'^\s*Progress:\s*(.+)$', reply, re.M)
        if found:
            progress.append((obs.elapsed_s, found.group(1).strip()[:40]))
        return parse(reply, info['width'], info['height'])


In [ ]:
# Plays one sample game here, live, and shows what the agent does. Change the three settings and rerun this cell.
import queue
from collections import Counter
from IPython.display import HTML, Image as Frame, display
from gga.game import Game

GAME = 'cavebots'             # asteroid_capitalist, blarglike, cavebots or off_line
PLAY_SECONDS = 120             # in the competition a game runs for 10 minutes
AGENT_VIEW = False             # False: the live game, every frame; True: only the screenshots the agent sees

# The competition's run skips this: the server in the last cell must be up within 15 minutes.
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    game = Game(GAME, video_dir='/kaggle/working')   # also saves a recording to the output files
    game.initialize({'minutes': PLAY_SECONDS / 60})
    tools = game.tools()
    agent = Agent()
    Agent.replies.clear()

    screen = display(HTML('<i>Starting...</i>'), display_id=True)
    status = display(HTML(''), display_id=True)
    live = queue.Queue()   # each turn: (when it came, its frames as (elapsed_s, jpeg)); None ends play_live

    def play_live():
        shown = last = None
        while (turn := live.get()) is not None:
            at, batch = turn
            step = (at - last) / len(batch) if last else 0.05
            last, start = at, time.time()
            for i, (elapsed, jpeg) in enumerate(batch, 1):
                if shown is None or elapsed > shown:
                    screen.update(Frame(data=jpeg, width=800))
                    shown = elapsed
                if not live.qsize():
                    time.sleep(max(0, start + i * step - time.time()))

    player = threading.Thread(target=play_live)
    player.start()

    turns, refused, step_s, tools_used = 0, [], [], Counter()
    while not game.is_terminal():
        obs = game.get_observation()
        if AGENT_VIEW:
            screen.update(Frame(data=obs.image, width=800))
        else:
            live.put((time.time(), obs.history + obs.frames))
        began = time.time()
        action = agent.step(obs, tools)
        step_s.append(time.time() - began)
        result = game.handle_tool(action.tool, action.params)
        turns += 1
        tools_used[('controller: ' if agent.controller else '') + action.tool] += 1
        if not result['ok'] and not game.is_terminal():
            refused.append(result.get('error'))
        last_event = agent.events[-1] if agent.events else ''
        status.update(HTML(f'Turn {turns}, game time {obs.elapsed_s:.0f} s, '
                           f'controller {"on" if agent.controller else "off"}, last event: <code>{last_event}</code>'))

    game.stop()
    live.put(None)
    player.join()
    print(f'{turns} turns, step() max {max(step_s) * 1000:.0f} ms, {len(Agent.replies)} Gemma replies, '
          f'{len(refused)} refused {refused[:3]}')
    print('Actions:', dict(tools_used))
    print('Events:', agent.events)
    with open('/kaggle/working/replies.txt', 'w') as f:   # every Gemma reply, to read after the run
        for elapsed, event, reply, secs, tokens in Agent.replies:
            f.write(f'=== {elapsed:.1f} s, {event} ({secs:.1f} s, {tokens} tokens)\n{reply}\n\n')
    for elapsed, event, reply, secs, tokens in Agent.replies[-3:]:
        print(f'\n=== {elapsed:.1f} s, {event} ({secs:.1f} s, {tokens} tokens)\n{reply}')


In [ ]:
# Hands the agent to the competition. Save & Run All, then submit that version.
from kaggle_evaluation.gga.gga_inference_server import GGAInferenceServer

Path('/kaggle/working/agent.py').write_text('from __main__ import Agent\n')  # the server loads the agent from this file
GGAInferenceServer().run()
